# Ilham on a free Kaggle GPU — brain **and** natural voice

Runs **Ollama** (her brain) on Kaggle's free dual T4, and an **edge-tts** server (her natural Urdu/English
voice) — both exposed through free Cloudflare tunnels, so a slow PC can use a big model and a real voice.

**Before running:** right panel → *Accelerator* = **GPU T4 x2**, *Internet* = **On** (needs a phone-verified
Kaggle account).

Run the cell, wait ~2 min, then copy the two printed blocks into Ilham's **⌨ Console** on your PC:
```
/set ollamaurl https://...trycloudflare.com      # her brain
/set model qwen2.5:14b
/set ttsurl   https://...trycloudflare.com        # her natural voice
```
Keep the cell running while you chat. Run **only one** Ilham against the tunnels at a time.

| Model | VRAM | Notes |
|---|---|---|
| `qwen2.5:7b`  | ~6 GB  | fast, safe fallback |
| `qwen2.5:14b` | ~10 GB | **recommended** — sharp Urdu + reasoning, one T4 |
| `qwen2.5:32b` | ~20 GB | smartest; spans both T4s |

The voice is `ur-PK-UzmaNeural` (a real Pakistani Urdu female voice) for Urdu/mix, and `en-US-AriaNeural` for English.


In [ ]:
import os, re, io, time, asyncio, threading, subprocess

MODEL = 'qwen2.5:14b'   # 'qwen2.5:7b' faster, 'qwen2.5:32b' smartest (both T4s)

# ---- 1. System deps + Ollama + edge-tts ------------------------------------------------
subprocess.run('apt-get -qq update && apt-get -qq install -y zstd', shell=True, check=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run('pip install -q edge-tts aiohttp', shell=True, check=True)

# ---- 2. Start Ollama, pull + pre-warm the model ----------------------------------------
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_ORIGINS='*',
           OLLAMA_KEEP_ALIVE='24h', OLLAMA_SCHED_SPREAD='1')
subprocess.Popen(['ollama', 'serve'], env=env)
time.sleep(6)
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Warming model into the GPUs...')
subprocess.run(['ollama', 'run', MODEL, 'Say ready.'], check=True)
print('Model warm:', MODEL)

# ---- 3. Voice server: POST /tts {text, voice, rate} -> MP3 audio -----------------------
import edge_tts
from aiohttp import web

async def _tts(request):
    try:
        d = await request.json()
    except Exception:
        return web.Response(status=400, text='bad json')
    text = (d.get('text') or '').strip()
    voice = d.get('voice') or 'ur-PK-UzmaNeural'
    rate = d.get('rate') or '+0%'
    if not re.fullmatch(r'[+-]\d{1,3}%', rate):
        rate = '+0%'
    if not text:
        return web.Response(status=400, text='no text')
    buf = io.BytesIO()
    try:
        async for chunk in edge_tts.Communicate(text, voice, rate=rate).stream():
            if chunk['type'] == 'audio':
                buf.write(chunk['data'])
    except Exception as e:
        return web.Response(status=502, text=f'tts failed: {e}')
    return web.Response(body=buf.getvalue(), content_type='audio/mpeg')

def _run_tts_server():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
    app = web.Application()
    app.router.add_post('/tts', _tts)
    app.router.add_get('/', lambda r: web.Response(text='ilham-tts ok'))
    runner = web.AppRunner(app); loop.run_until_complete(runner.setup())
    loop.run_until_complete(web.TCPSite(runner, '0.0.0.0', 8080).start())
    print('Voice server listening on :8080')
    loop.run_forever()

threading.Thread(target=_run_tts_server, daemon=True).start()
time.sleep(2)

# ---- 4. Two Cloudflare tunnels: one for the brain (11434), one for the voice (8080) ----
subprocess.run('curl -sL -o cloudflared '
               'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
               '&& chmod +x cloudflared', shell=True, check=True)

def start_tunnel(port):
    p = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://localhost:{port}', '--no-autoupdate'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in p.stdout:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0); break
    threading.Thread(target=lambda: [None for _ in p.stdout], daemon=True).start()  # drain logs
    return url

brain_url = start_tunnel(11434)
voice_url = start_tunnel(8080)

print('\n==================================================')
print('  Paste into Ilham Console on your PC:')
print('     /set ollamaurl ' + str(brain_url))
print('     /set model ' + MODEL)
print('     /set ttsurl ' + str(voice_url))
print('     /model')
print('  Keep this cell running. One Ilham at a time.')
print('==================================================\n')

while True:
    time.sleep(60)
